# Train Squad's own model on a free Colab GPU

Fine-tunes Qwen3.5-0.8B with LoRA to route jobs to Squad's crew, scores it on the fixed test set, and gives you one ~0.5 GB `fin-q4_k_m.gguf` file to download.

**First:** Runtime → Change runtime type → **T4 GPU** → Save. Then Runtime → Run all. It takes about 30 to 45 minutes.

No API key is needed: the examples come from the hand-written seeds in `training/seeds/`.

In [23]:
# 1. The GPU and the code
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, getpass
REPO = "github.com/cosmospan/squadcode"
BRANCH = "claude/project-thread-8o4ttl"
os.chdir("/content")
if not os.path.isdir("squadcode"):
    r = os.system(f"git clone -q --depth 1 -b {BRANCH} https://{REPO} squadcode")
    if r != 0:  # a private repo needs a GitHub token with read access
        token = getpass.getpass("GitHub token (read access to the repo): ")
        os.system(f"git clone -q --depth 1 -b {BRANCH} https://x-access-token:{token}@{REPO} squadcode")
%cd /content/squadcode/training

name, memory.total [MiB]
Tesla T4, 15360 MiB
/content/squadcode/training


In [24]:
# 2. Packages. Colab's own torchao is too old for the newest PEFT.
!pip install -q -U transformers peft trl datasets accelerate "torchao>=0.16.0"

In [25]:
# 3. Training examples from the seeds (no API key)
!python build_data.py --out work/data/synth.jsonl

ask: 327, build: 410, fix: 427, review: 249, test: 225, tidy: 221 -> 1859 examples in work/data/synth.jsonl


In [26]:
!git -C /content/squadcode pull && git -C /content/squadcode log --oneline -1

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 8 (delta 6), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 1.54 KiB | 316.00 KiB/s, done.
From https://github.com/cosmospan/squadcode
   59cda37..24716ac  claude/project-thread-8o4ttl -> origin/claude/project-thread-8o4ttl
Updating 59cda37..24716ac
Fast-forward
 training/train_lora.py | 38 +++++++++++++++++++++++++++++---------
 1 file changed, 29 insertions(+), 9 deletions(-)
24716ac (HEAD -> claude/project-thread-8o4ttl, origin/claude/project-thread-8o4ttl) train_lora.py: bfloat16 only on Ampere or newer; is_bf16_supported() counts a T4's emulation


In [27]:
# 4. Train (about 15 to 30 minutes on a T4)
!python train_lora.py --data work/data/synth.jsonl --out work/fin-lora

Older GPU: TRL's loss widens to float32 before multiplying.
Loading weights: 100% 320/320 [00:03<00:00, 106.44it/s]
Generating train split: 1859 examples [00:00, 20633.16 examples/s]
Map: 100% 1859/1859 [00:00<00:00, 2971.97 examples/s]
LoRA on: down_proj, gate_proj, in_proj_a, in_proj_b, in_proj_qkv, in_proj_z, k_proj, o_proj, out_proj, q_proj, up_proj, v_proj
Tokenizing train dataset: 100% 1859/1859 [00:04<00:00, 421.32 examples/s]
Building labels for train dataset: 100% 1859/1859 [00:00<00:00, 4894.35 examples/s]
Truncating train dataset: 100% 1859/1859 [00:00<00:00, 8151.96 examples/s]
Dropping fully masked examples from train dataset: 100% 1859/1859 [00:00<00:00, 26800.80 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
  0% 0/118 [00:00<?, ?it/s][transformer

In [29]:
# 5. Score it on the fixed test set, as the app asks it
import json, os, sys, torch
# Colab's Python may not be in the training folder after a restart.
os.chdir("/content/squadcode/training")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
from transformers import AutoModelForCausalLM, AutoTokenizer
from make_data import route_system
tok = AutoTokenizer.from_pretrained("work/fin-lora/merged")
model = AutoModelForCausalLM.from_pretrained("work/fin-lora/merged", torch_dtype=torch.float16).cuda().eval()
system = route_system()
cases = [json.loads(l) for l in open("eval/route.jsonl") if l.strip()]
right, missed = 0, []
for c in cases:
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": c["text"]}]
        ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True, enable_thinking=False)["input_ids"].cuda()
    out = model.generate(ids, max_new_tokens=80, do_sample=False)
    reply = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    try:
        got = json.loads(reply[reply.find("{"): reply.rfind("}") + 1])["mode"]
    except Exception:
        got = "?"
    if got == c["mode"]:
        right += 1
    else:
        missed.append((c["text"], c["mode"], got))
print(f"Model: {right}/{len(cases)}  (the rules get 51/53)")
for m in missed:
    print("  missed:", m)

IndentationError: unexpected indent (2735800717.py, line 16)

In [ ]:
# 6. Shrink to one GGUF file (about 0.5 GB)
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp work/llama.cpp
!pip install -q -r work/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python work/llama.cpp/convert_hf_to_gguf.py work/fin-lora/merged --outfile work/fin-f16.gguf --outtype f16
!cmake -S work/llama.cpp -B work/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build work/llama.cpp/build --target llama-quantize -j 4 > /dev/null
!work/llama.cpp/build/bin/llama-quantize work/fin-f16.gguf work/fin-q4_k_m.gguf Q4_K_M
!ls -lh work/fin-q4_k_m.gguf

In [ ]:
# 7. Download it
from google.colab import files
files.download("work/fin-q4_k_m.gguf")

## On your Mac

Put the file somewhere (say `~/Downloads`), then:

```
brew install llama.cpp
llama-server -m ~/Downloads/fin-q4_k_m.gguf --port 8088 --alias fin
```

In `~/.squadcode/settings.json` set `"brain_model": "fin"`, and Squad uses it. `SquadCode v2 brain eval` scores it against the rules.